# Midterm 2 — Integrated Evidence Notebook

This notebook consolidates **already-saved outputs** from the project repository. It does not train or refit models.

Expected location: save this notebook under `notebooks/` in the main repository, or run it from the repository root. All source paths are repository-relative.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "reports").is_dir() and (p / "scripts").is_dir())
BASE = ROOT / "reports" / "tables" / "midterm2-20261104"
W4 = ROOT / "reports" / "tables" / "week4-comparison-20261111"
H1 = ROOT / "reports" / "tables" / "h1-ablation-20261109"
W3 = ROOT / "reports" / "tables" / "week3-review-20261109"
OUT = ROOT / "reports" / "combined-midterm2-20261111"
OUT.mkdir(parents=True, exist_ok=True)

def read_csv(path):
    path = ROOT / path
    if not path.exists():
        raise FileNotFoundError(f"Required saved output not found: {path}")
    return pd.read_csv(path, encoding="utf-8-sig")

def read_json(path):
    path = ROOT / path
    if not path.exists():
        raise FileNotFoundError(f"Required saved output not found: {path}")
    return json.loads(path.read_text(encoding="utf-8"))

print("Repository:", ROOT)
print("Output directory:", OUT)

## 1. Load and validate saved full-study outputs

In [ ]:
a1_overall = read_csv("reports/tables/midterm2-20261104/A1/tables/A1_overall_macro_summary.csv")
a1_class = read_csv("reports/tables/midterm2-20261104/A1/tables/A1_overall_class_metrics.csv")
a1_month = read_csv("reports/tables/midterm2-20261104/A1/tables/A1_monthly_macro_summary.csv")
a2_strata = read_csv("reports/tables/midterm2-20261104/A2/tables/A2_stratified_macro_summary.csv")
a3_numeric = read_csv("reports/tables/midterm2-20261104/A3/tables/A3_numeric_comparison.csv")
a3_size = read_csv("reports/tables/midterm2-20261104/A3/tables/A3_status_by_size_band.csv")
a3_history = read_csv("reports/tables/midterm2-20261104/A3/tables/A3_status_by_history.csv")
b_month = read_csv("reports/tables/midterm2-20261104/B/tables/B_monthly_coverage.csv")
b_size = read_csv("reports/tables/midterm2-20261104/B/tables/B_size_coverage.csv")
b_history = read_csv("reports/tables/midterm2-20261104/B/tables/B_history_coverage.csv")
errors = read_csv("reports/tables/midterm2-20261104/error_analysis/tables/error_by_true_class.csv")
error_pairs = read_csv("reports/tables/midterm2-20261104/error_analysis/tables/misclassification_pairs.csv")

# Fixed-cohort and status-count checks
assert set(a1_overall["seed"].unique()) == {42, 43, 44}
assert a1_overall["n"].nunique() == 1 and int(a1_overall["n"].iloc[0]) == 1269
assert int(a3_size[["resolved", "unresolved", "censored"]].to_numpy().sum()) == 11745
assert int(a3_history[["resolved", "unresolved", "censored"]].to_numpy().sum()) == 11745

status_totals = {
    "resolved": int(a3_size["resolved"].sum()),
    "unresolved": int(a3_size["unresolved"].sum()),
    "censored": int(a3_size["censored"].sum()),
}
coverage = status_totals["resolved"] / (status_totals["resolved"] + status_totals["unresolved"])
print(status_totals, "community-month coverage =", round(coverage, 6))

## 2. A1 — Overall, class and monthly performance

In [ ]:
overall = (a1_overall.groupby("model", as_index=False)
           .agg(macro_f1=("macro_f1_six", "mean"),
                accuracy=("accuracy", "mean"),
                n_per_seed=("n", "max"),
                seeds=("seed", "nunique"))
           .sort_values("macro_f1", ascending=False))
display(overall)
overall.to_csv(OUT / "A1_overall_model_summary.csv", index=False)

proposed_class = a1_class[(a1_class.model == "hier_lr_dynamic") & (a1_class.seed == 42)].copy()
display(proposed_class[["label","support","precision","recall","f1","to_dissolve","predicted"]])
proposed_class.to_csv(OUT / "A1_proposed_per_class_seed42.csv", index=False)

proposed_month = (a1_month[a1_month.model == "hier_lr_dynamic"]
                  .groupby("stratum", as_index=False)
                  .agg(n=("n","max"), macro_f1=("macro_f1_six","mean"), accuracy=("accuracy","mean")))
display(proposed_month)
proposed_month.to_csv(OUT / "A1_proposed_monthly_summary.csv", index=False)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
plot_df = overall.head(10).sort_values("macro_f1")
ax.barh(plot_df["model"], plot_df["macro_f1"])
ax.set_xlabel("Six-class Macro-F1")
ax.set_title("Full-study final-window model comparison")
fig.tight_layout()
fig.savefig(OUT / "A1_overall_macro_f1.png", dpi=160)
plt.show()

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(proposed_month["stratum"], proposed_month["macro_f1"], marker="o")
ax.set_xlabel("Label month")
ax.set_ylabel("Six-class Macro-F1")
ax.set_title("hier_lr_dynamic monthly Macro-F1")
ax.tick_params(axis="x", rotation=45)
fig.tight_layout()
fig.savefig(OUT / "A1_monthly_macro_f1.png", dpi=160)
plt.show()

## 3. A2 — Size and history strata

In [ ]:
proposed_strata = a2_strata[(a2_strata.model == "hier_lr_dynamic") & (a2_strata.seed == 42)].copy()
display(proposed_strata[["axis","stratum","n","macro_f1_six","accuracy"]])
proposed_strata.to_csv(OUT / "A2_proposed_strata_seed42.csv", index=False)

## 4. A3/B — Coverage and resolved/unresolved comparisons

In [ ]:
coverage_by_size = b_size.copy()
coverage_by_history = b_history.copy()
display(coverage_by_size)
display(coverage_by_history)
coverage_by_size.to_csv(OUT / "B_coverage_by_size.csv", index=False)
coverage_by_history.to_csv(OUT / "B_coverage_by_history.csv", index=False)

# Show selected distribution summaries; no inferential tests are performed here.
selected_vars = ["size", "density", "community_age", "past_growth", "past_retention",
                 "inflow_fraction", "outflow_fraction", "thread_activity",
                 "active_member_fraction", "context_drift_js"]
resolved_unresolved = a3_numeric[
    a3_numeric["variable"].isin(selected_vars) &
    a3_numeric["status"].isin(["resolved", "unresolved"])
][["status","variable","n_rows","mean","median","q25","q75","missing_rate"]]
display(resolved_unresolved)
resolved_unresolved.to_csv(OUT / "A3_resolved_unresolved_selected_features.csv", index=False)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(b_month["label_month"], b_month["coverage"], marker="o", markersize=3)
ax.set_ylim(0, 1)
ax.set_ylabel("Resolved / observed transitions")
ax.set_xlabel("Label month")
ax.set_title("Monthly community-month label coverage")
ax.tick_params(axis="x", rotation=75)
fig.tight_layout()
fig.savefig(OUT / "B_monthly_coverage.png", dpi=160)
plt.show()

## 5. Error analysis

In [ ]:
display(errors)
display(error_pairs.head(15))
errors.to_csv(OUT / "error_by_true_class.csv", index=False)
error_pairs.to_csv(OUT / "misclassification_pairs.csv", index=False)

## 6. Week 4 — Proposed versus HGB and baseline comparison

In [ ]:
w4_baselines = read_csv("reports/tables/week4-comparison-20261111/primary_baseline_comparison.csv")
w4_class = read_csv("reports/tables/week4-comparison-20261111/per_class_metrics.csv")
w4_pairs = read_csv("reports/tables/week4-comparison-20261111/paired_wins_losses_summary.csv")
w4_by_class = read_csv("reports/tables/week4-comparison-20261111/paired_wins_losses_by_class.csv")
w4_recon = read_json("reports/tables/week4-comparison-20261111/reconciliation_summary.json")
display(w4_baselines)
display(w4_pairs[w4_pairs.comparator_model == "hier_hgb_dynamic"])
display(w4_class[w4_class.model.isin(["hier_lr_dynamic","hier_hgb_dynamic"])])
assert w4_recon["status"] == "PASS"
assert w4_recon["both_correct"] == 1119
assert w4_recon["proposed_only"] == 29
assert w4_recon["hgb_only"] == 35
assert w4_recon["both_wrong"] == 86

## 7. Human review, tracker and lineage

In [ ]:
review_summary = read_json("reports/tables/week3-review-20261109/review-summary.json")
tracker_confusion = read_csv("reports/tables/week3-review-20261109/review-tracker-confusion.csv")
lineage_summary = read_csv("reports/tables/week3-review-20261109/source-lineage-summary.csv")
print(json.dumps(review_summary, indent=2))
display(tracker_confusion)
display(lineage_summary)

## 8. H1 — Separate 2015 HeTCo pilot (do not merge with full-study scores)

In [ ]:
h1_scores = read_csv("reports/tables/h1-ablation-20261109/ablation.csv")
h1_contrasts = read_csv("reports/tables/h1-ablation-20261109/contrasts.csv")
h1_strata = read_csv("reports/tables/h1-ablation-20261109/main-strata.csv")
h1_audit = read_json("reports/tables/h1-ablation-20261109/verification.json")
display(h1_scores)
display(h1_contrasts)
assert h1_audit["status"] == "verified"
assert h1_audit["decision"] == "H1 not supported; v4 retained; v5 reported as ablation"

In [ ]:
h1_main = h1_scores[h1_scores.window == "main"].set_index("model")
v4 = h1_main.loc["HeTCo_v4"]
v5 = h1_main.loc["HeTCo_v5"]
print("Main Macro-F1 delta:", round(v5.macro_f1 - v4.macro_f1, 4))
print("Missed events v4 -> v5:", v4.missed_events, "->", v5.missed_events)
print("False events v4 -> v5:", round(v4.false_events, 1), "->", round(v5.false_events, 1))

## 9. Claim–Evidence ledger and export

In [ ]:
claims = pd.DataFrame([
    ["Coverage is incomplete", "7,490 resolved; 4,072 unresolved; 183 censored; coverage 64.78%", "A3/B coverage tables"],
    ["Proposed has highest primary Macro-F1", "hier_lr_dynamic 0.4117 vs hier_hgb_dynamic 0.3880", "A1 overall; Week 4 baseline comparison"],
    ["Proposed does not win paired correctness vs HGB", "29 proposed-only correct vs 35 HGB-only correct", "Week 4 paired_wins_losses_summary.csv"],
    ["Grow/merge remain difficult", "Recall 0.1579 / 0.2000; small supports 19 / 15", "A1 class metrics; Week 4 per_class_metrics.csv"],
    ["Reviewer submissions show moderate descriptive agreement", "47/60; raw agreement 78.33%; kappa 0.4433", "Review handoff summary.json"],
    ["H1 is not supported in its predicted direction", "Missed events 18 -> 22; overlap tier-1 recall 0.651 -> 0.508; paired interval includes zero", "H1 report; ablation.csv; saved-intervals.json; main-strata.csv"],
], columns=["claim","evidence_summary","source"])
display(claims)
claims.to_csv(OUT / "claim_evidence_table.csv", index=False)
print("Saved integrated tables and figures to:", OUT)